In [20]:
pip install pandas lxml osmnx

Note: you may need to restart the kernel to use updated packages.


In [21]:
import time
import osmnx as ox
import pandas as pd
import geopandas as gpd

In [22]:
# Parametros
ox.settings.use_cache = True
ox.settings.requests_timeout = 300

LUGAR = "Providencia, Santiago, Chile"

categorias = {
    "Salud": {
        "Hospital":        {"amenity": "hospital"},
        "Clinica":         {"amenity": "clinic"},
        "Centro_salud":    {"healthcare": "centre"},          # CESFAM, etc.
        "Consultorio":     {"amenity": "doctors"},
        "Farmacia":        {"amenity": "pharmacy"},
    },
    "Recreativo_Cultural": {
        "Parque":          {"leisure": "park"},
        "Mall":            {"shop": "mall"},
        "Restaurante":     {"amenity": "restaurant"},
        "Museo":           {"tourism": "museum"},
        "Biblioteca":      {"amenity": "library"},
        "Teatro":          {"amenity": "theatre"},
    },
    "Seguridad": {
        "Bomberos":        {"amenity": "fire_station"},
        "Comisaria":       {"amenity": "police"},
    },
    "Educacion": {
        "Colegio":         {"amenity": "school"},
        "Jardin_infantil": {"amenity": "kindergarten"},
        "Universidad":     {"amenity": "university"},
    },
    "Comercio": {
        "Supermercado":    {"shop": "supermarket"},
        "Banco":           {"amenity": "bank"},
        "Feria":           {"amenity": "marketplace"},
        "Almacen":         {"shop": ["convenience", "general", "greengrocer"]},
    },
    "Transporte": {
        "Metro_estacion":  {"station": "subway"},
        "Metro_railway":   {"railway": "station"},            # incluye EFE; filtrar luego
        "Metro_entrada":   {"railway": "subway_entrance"},
        "Paradero_bus":    {"highway": "bus_stop"},
        "Paradero_plataforma": {"public_transport": "platform"},
        "Terminal_bus":    {"amenity": "bus_station"},
    },
    "Negativos": {
        "Vertedero":       {"landuse": "landfill"},
        "Planta_residuos": {"amenity": "waste_transfer_station"},
        "Industrial":      {"landuse": "industrial"},
        "Fabrica":         {"man_made": "works"},
        "Botilleria":      {"shop": "alcohol"},
        "Bar_pub":         {"amenity": ["bar", "pub", "nightclub"]},
        "Sitio_eriazo":    {"landuse": ["brownfield", "greenfield"]},
        "Suelo_desnudo":   {"natural": "bare_rock"},
    },
}


In [23]:

# 1. Unir todos los tags en un solo diccionario {clave: [valores]}
tags_unidos = {}
for subtipos in categorias.values():
    for tags in subtipos.values():
        for k, v in tags.items():
            vals = v if isinstance(v, list) else [v]
            tags_unidos.setdefault(k, set()).update(vals)
tags_unidos = {k: sorted(v) for k, v in tags_unidos.items()}

In [24]:

# 2. Geocodificar UNA vez y descargar UNA vez (midiendo tiempos)
t0 = time.perf_counter()
limite = ox.geocode_to_gdf(LUGAR)
poligono = limite.geometry.iloc[0]
print(f"Geocodificación: {time.perf_counter() - t0:.1f}s")

t0 = time.perf_counter()
crudo = ox.features_from_polygon(poligono, tags=tags_unidos).reset_index()
crudo = crudo.rename(columns={"element": "osm_type", "id": "osm_id"})
print(f"Descarga Overpass: {time.perf_counter() - t0:.1f}s ({len(crudo)} elementos)")

Geocodificación: 1.1s
Descarga Overpass: 125.5s (1633 elementos)


In [25]:
# 3. Clasificar localmente por categoría/tipo (instantáneo)
t0 = time.perf_counter()
resultados, resumen = [], []
for categoria, subtipos in categorias.items():
    for tipo, tags in subtipos.items():
        mask = pd.Series(False, index=crudo.index)
        for k, v in tags.items():
            if k in crudo.columns:
                vals = v if isinstance(v, list) else [v]
                mask |= crudo[k].isin(vals)
        sub = crudo[mask].copy()
        sub["categoria"], sub["tipo"] = categoria, tipo
        resultados.append(sub)
        resumen.append((categoria, tipo, len(sub)))
print(f"Clasificación: {time.perf_counter() - t0:.1f}s")

datos = gpd.GeoDataFrame(pd.concat(resultados, ignore_index=True),
                         geometry="geometry", crs="EPSG:4326")
df_resumen = pd.DataFrame(resumen, columns=["categoria", "tipo", "n"])

Clasificación: 0.3s


In [26]:
# print(df_resumen)

# Coordenadas representativas
cent = datos.to_crs(32719).centroid.to_crs(4326)
datos["lon"], datos["lat"] = cent.x, cent.y
datos["geom_tipo"] = datos.geom_type

# Guardado (sufijo _test)
datos.to_file("providencia_features_test.gpkg", layer="features", driver="GPKG")
limite.to_file("providencia_features_test.gpkg", layer="limite", driver="GPKG")
datos.drop(columns="geometry").to_csv("providencia_features_test.csv", index=False)
df_resumen.to_csv("resumen_conteos_test.csv", index=False)